In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from itertools import chain
import nltk
from scipy import stats
import os

# Loading dataset

In [ ]:
full_dataset_path = "datasets/one_tweet_dataset_full.csv"
if os.path.exists(full_dataset_path):
    df = pd.read_csv(full_dataset_path)
else:
    print("Dataset not found. Please ensure the dataset is located at '{}'.".format(full_dataset_path))

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.nunique().sort_values(ascending=False)

# Extracting data from text

## Tokenization for lexical diversity

### Create clean tokens function

In [ ]:
def create_clean_tokens(text: str, stop_words: list[str] = ENGLISH_STOP_WORDS, topic_words: list[str] = []) -> str|None:
    if not text:
        return None
    
    text = text.lower().strip()
    
    if text.startswith('@'):
        return None
    if text.startswith('#'):
        return None
    if text.startswith('&'):
        return None
    if text.startswith('http'):
        return None

    if text.isdigit():
        return None
    if len(text) < 2:
        return None
    
    text = re.sub(r"[^a-z]", "", text)
    if text.startswith('@'):
        return None
    if text.startswith('#'):
        return None
    if text.startswith('&'):
        return None
    if text.startswith('http'):
        return None
    
    if text in stop_words:
        return None
    if text in topic_words:
        return None

    return text

def tokenize_clear(text: str, topic_words: list[str] = []) -> list[str]:
    if not isinstance(text, str):
        return []

    words = text.split()
    cleaned = [create_clean_tokens(w, topic_words=topic_words) for w in words]
    return [w for w in cleaned if w]

### Create clean tokens

In [ ]:
df['clean_tokens'] = df['text'].apply(tokenize_clear, args=(['vaccine','covid','pandemic', 'vaccines', 'vaccination', 'virus', 'vaccinated', 'coronavirus'],))

In [ ]:
df.head()

### Create dirty tokens function

In [ ]:
from math import e


emoji_re = re.compile("["
                        u"\U0001F600-\U0001F64F"   
                        u"\U0001F300-\U0001F5FF"
                        u"\U0001F680-\U0001F6FF"
                        u"\U0001F1E0-\U0001F1FF"
                        u"\U00002702-\U000027B0"  
                        u"\U000024C2-\U0001F251"
                        "]", flags=re.UNICODE)
emoji_unique = set(chain.from_iterable(df['text'].str.findall(emoji_re)))
len(emoji_unique), emoji_unique

In [ ]:
emoji_enum = {emoji: f"emoji_{idx}" for idx, emoji in enumerate(emoji_unique)}
emoji_enum

In [ ]:
def create_dirty_tokens(text: str, topic_words: list[str] = []):
    emoji_re = re.compile("["
                        u"\U0001F600-\U0001F64F"   
                        u"\U0001F300-\U0001F5FF"
                        u"\U0001F680-\U0001F6FF"
                        u"\U0001F1E0-\U0001F1FF"
                        u"\U00002702-\U000027B0"  
                        u"\U000024C2-\U0001F251"
                        "]+", flags=re.UNICODE)
    
    tokens: list[str] = text.split()
    filtered =  []
    for token in tokens:
        if token.lower() in topic_words:
            continue
        if token.strip().lower() in topic_words:
            continue
        emoji_match = emoji_re.search(token)
        if emoji_match:
            emoji_char = emoji_match.group()
            token = token.replace(emoji_char, emoji_enum.get(emoji_char, emoji_char))

        token = token.replace(' ', '_')
        found = False
        for item in topic_words:
            if token.lower().find(item) != -1:
                found = True
                break
        if found:
            continue

        if token in {'http','https','co','t','amp','gt','rt'}:
            continue
        for item in {'http','https','co','t','amp','gt','rt'}:
            if token.lower().find(item) != -1:
                found = True
                break
        if found:
            continue

        filtered.append(token)

    return filtered

### Create dirty tokens

In [ ]:
topic_words = ['vaccine','covid','pandemic', 'vaccines', 'vaccination', 'virus', 'vaccinated', 'coronavirus', '19', 'covid']

In [ ]:
df['dirty_tokens'] = df['text'].apply(lambda x: create_dirty_tokens(x, topic_words=topic_words))

In [ ]:
df.head()

## Lenght

### Length of text

In [ ]:
df['char_len'] = df['text'].apply(len)

In [ ]:
df.head()

check how many tokens have less then 20 chars

In [ ]:
df[df['char_len'] <= 20].head(10).sort_values(by='char_len', ascending=True)

In [ ]:
df[df['char_len'] <= 20]['char_len'].count()

### Clean tokens count

In [ ]:
df['tokens_count_clean'] = df['clean_tokens'].apply(len)

check what least token count

In [ ]:
df[df['tokens_count_clean'] <= 5].head(10).sort_values(by='char_len', ascending=True)

how much rows have token count less than 5

In [ ]:
df[df['tokens_count_clean'] <= 5]['tokens_count_clean'].count()

### Dirty tokens count

In [ ]:
df['tokens_count_dirty'] = df['dirty_tokens'].apply(len)

In [ ]:
df['tokens_count_dirty'].value_counts(ascending=False)

## Features extraction

### Function to extract text features

In [ ]:
def features_extraction(text: str) -> dict:
    text = text.lower()
    P_PERIOD: set[str] = {'.'}
    P_COMMA = {','}
    P_EXCL  = {'!'}
    P_QMARK = {'?'}
    P_QUOTE = {'"', '«', '»', '“', '”', '„', "'"}

    emoji_re = re.compile("["
                        u"\U0001F600-\U0001F64F"  # emoticons
                        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                        u"\U0001F680-\U0001F6FF"  # transport & map symbols
                        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                        "]+", flags=re.UNICODE)
    mention_re = re.compile(r'^@\w+$')
    hashtag_re = re.compile(r'^#\w+$')

    n_chars = len(text)
    tokens = text.split()
    tokens_count = len(tokens)
    ttr = (len(set(tokens)) / tokens_count) if tokens_count else 0.0
    hapax_ratio = (sum(1 for t in set(tokens) if tokens.count(t) == 1) / tokens_count) if tokens_count else 0.0
    avg_token_len = (np.mean([len(t) for t in tokens]) if tokens_count else 0.0)
    emojis = emoji_re.findall(text)
    mentions = [t for t in tokens if mention_re.match(t)]
    hashtags = [t for t in tokens if hashtag_re.match(t)]
    periods = sum(text.count(p) for p in P_PERIOD)
    commas = sum(text.count(p) for p in P_COMMA)
    excls = sum(text.count(p) for p in P_EXCL)
    qmarks = sum(text.count(p) for p in P_QMARK)
    quotes = sum(text.count(p) for p in P_QUOTE)

    per100 = (lambda x: (x / tokens_count) * 100) if tokens_count else (lambda x: 0.0)

    return {
        'len_tokens': tokens_count  ,
        'ttr': ttr,
        'hapax_ratio': hapax_ratio,
        'avg_token_len': avg_token_len,
        'emojis': len(emojis),
        'mentions': len(mentions),
        'hashtags': len(hashtags),
        'periods': periods,
        'commas': commas,
        'excls': excls,
        'qmarks': qmarks,
        'quotes': quotes,
        'emojis_per100': per100(len(emojis)),
        'mentions_per100': per100(len(mentions)),
        'hashtags_per100': per100(len(hashtags)),
        'periods_per100': per100(periods),
        'commas_per100': per100(commas),
        'excls_per100': per100(excls),
        'qmarks_per100': per100(qmarks),
        'quotes_per100': per100(quotes),
    }

### Using the function to extract features

In [ ]:
features_df = df['text'].apply(features_extraction).apply(pd.Series)
df = pd.concat([df, features_df], axis=1)
df.head()

# Analyse data with plots and statistics

## Gender distribution

all data

In [ ]:
df.shape[0]

data by gender

In [ ]:
df['gender_label'].value_counts()

gender distribution in dataset

In [ ]:
plt.figure(figsize=(5,5))
counts = df['gender_label'].value_counts()
plt.pie(
    counts,
    autopct='%1.1f%%',
    startangle=90,
    labels=counts.index.tolist(),
)
plt.legend()
plt.title("Gender distribution (%)")
plt.show()

gender distribution but with cahr_len more than 20

In [ ]:
plt.figure(figsize=(5,5))
counts = df.loc[df['char_len'] > 20, 'gender_label'].value_counts()
plt.pie(
    counts,
    autopct='%1.1f%%',
    startangle=90,
    labels=counts.index.tolist(),
)
plt.legend()
plt.title("Gender distribution (%)")
plt.show()

gender distribution but with token count more than 5

In [ ]:
plt.figure(figsize=(5,5))
counts = df.loc[df['tokens_count_clean'] > 5, 'gender_label'].value_counts()
plt.pie(
    counts,
    autopct='%1.1f%%',
    startangle=90,
    labels=counts.index.tolist(),
)
plt.legend()
plt.title("Gender distribution (%)")
plt.show()

## Summury statistics

In [ ]:
df.groupby('gender_label')[['char_len', 'tokens_count_clean']].agg(['mean', 'median', 'min', 'max', 'std'])

In [ ]:
cols = df.select_dtypes('number').columns.tolist()
fig, axes = plt.subplots(nrows=len(cols)//4 + 1, ncols=4, figsize=(20, 20))

for i, col in enumerate(cols):
    row = i // 4
    col_idx = i % 4
    sns.boxplot(data=df, x='gender_label', y=col, showfliers=False, ax=axes[row, col_idx])
    axes[row, col_idx].set_title(col)

for i in range(len(cols), len(axes.flat)):
    fig.delaxes(axes.flat[i])

plt.tight_layout()
plt.show()

In [ ]:
df.groupby('gender_label').mean(True)

## Lexical diversity statistics

We will use clear tokens without stopwords, emojis, mentions, hashtags, punctuation marks etc. Because we need to know lexical diversity

### Unique words

In [ ]:
all_words = set(chain.from_iterable(df['clean_tokens'].tolist()))
unique_word_count = len(all_words)
print(f"Total unique words: {unique_word_count}")

### Create gender df

In [ ]:
male_df = df[df['gender_label'] == 'M'].drop(columns=['gender_label'])
female_df = df[df['gender_label'] == 'F'].drop(columns=['gender_label'])

In [ ]:
male_df.info()

In [ ]:
female_df.info()

### Unique words used only by one gender

In [ ]:
all_words_male = Counter(chain.from_iterable(male_df['clean_tokens']))
all_words_female = Counter(chain.from_iterable(female_df['clean_tokens']))
male_only = {word: count for word, count in all_words_male.items() if word not in all_words_female}
female_only = {word: count for word, count in all_words_female.items() if word not in all_words_male}

print(f"Words used only by males: {len(male_only)}. {list(male_only)[:20]}")
print(f"Words used only by females: {len(female_only)}. {list(female_only)[:20]}")

In [ ]:
all_words_male.most_common(20)


In [ ]:
all_words_female.most_common(20)

### Histplot

just to see distribution of unique words used

In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(
    data=df,
    x="tokens_count_clean",
    hue="gender_label",
    bins=np.linspace(0, np.percentile(df["tokens_count_clean"], 99), 40),
    element="step",
    stat="density",
    common_norm=False,
    kde=True,
    palette={"M":"tab:blue", "F":"tab:red"}
)
plt.xlabel("Number of tokens in message")
plt.ylabel("Density")
plt.title("Distribution of message length by gender")
plt.tight_layout()
plt.show()

### Boxplot

for token count by gender

In [ ]:
sns.boxplot(x='gender_label', y='tokens_count_clean', data=df)

for char len by gender

In [ ]:
sns.boxplot(x='gender_label', y='char_len', data=df)

### Scatter plot

the most used words by gender

In [ ]:
male_word_freq = Counter(chain.from_iterable(male_df['clean_tokens']))
female_word_freq = Counter(chain.from_iterable(female_df['clean_tokens']))

In [ ]:
top_n = 50
male_top = male_word_freq.most_common(top_n)
female_top = female_word_freq.most_common(top_n)
print("Top {} male words: {}".format(top_n, male_top))
print("Top {} female words: {}".format(top_n, female_top))

In [ ]:
least_n = 10
male_least = male_word_freq.most_common()[:-least_n-1:-1]
female_least = female_word_freq.most_common()[:-least_n-1:-1]
print("Least {} male words: {}".format(least_n, male_least))
print("Least {} female words: {}".format(least_n, female_least))

In [ ]:
plt.figure(figsize=(12, 6))

male_words_list = [word for word, count in male_top]
male_counts = [count for word, count in male_top]
female_words_list = [word for word, count in female_top]
female_counts = [count for word, count in female_top]

plt.scatter(range(len(male_top)), male_counts, alpha=0.6, s=10, label='Male', color='red')
plt.scatter(range(len(female_top)), female_counts, alpha=0.6, s=10, label='Female', color='green')

for i in range(top_n):
    plt.annotate(male_words_list[i], (i, male_counts[i]), fontsize=6, alpha=0.7, color='blue')
    plt.annotate(female_words_list[i], (i, female_counts[i]), fontsize=6, alpha=0.7, color='black')

plt.title(f"Top {top_n} word frequencies by gender")
plt.xlabel("Word rank")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### qq-plot

shows us what word used with bigger probability by which gender

only words that are presented in both genders

In [ ]:
male_total = sum(male_word_freq.values())
female_total = sum(female_word_freq.values())
common_words = set(male_word_freq.keys()) & set(female_word_freq.keys())
words_plot = pd.DataFrame([
    {
        'word': word,
        'p_m': male_word_freq[word] / male_total,
        'p_f': female_word_freq[word] / female_total
    }
    for word in common_words
])

words_plot['male_q'] = words_plot['p_m'].apply(np.log1p).apply(lambda x: x * 10**3)
words_plot['female_q'] = words_plot['p_f'].apply(np.log1p).apply(lambda x: x * 10**3)
words_plot['diff'] = np.abs(words_plot['male_q'] - words_plot['female_q'])

top_20_words = words_plot.nlargest(20, 'diff')

plt.figure(figsize=(15,10))
sns.scatterplot(data=top_20_words, x="male_q", y="female_q", color="purple", alpha=0.6, s=30)
lims = [top_20_words[['male_q', 'female_q']].min().min(), top_20_words[['male_q', 'female_q']].max().max()]
sns.lineplot(x=lims, y=lims, color="gray", linestyle="--")
plt.xlabel("Quantiles of log1p word shares male")
plt.ylabel("Quantiles of log1p word shares female")
plt.title("QQ-plot of word-share distributions by gender")

for idx, row in top_20_words.iterrows():
    x_val = row['male_q']
    y_val = row['female_q']
    word = row['word']
    plt.annotate(word, (x_val, y_val), fontsize=8, alpha=0.7)

plt.show()

### Pairplot

In [ ]:
features_for_pairplot = [
    'char_len', 
    'tokens_count_clean', 
    'avg_token_len',
    'ttr',
    'emojis_per100',
    'hashtags_per100',
    'excls_per100',
    'gender_label'
]

plt.figure(figsize=(15, 15))
g = sns.pairplot(
    df[features_for_pairplot], 
    hue='gender_label',
    palette={'M': 'tab:blue', 'F': 'tab:red'},
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 's': 10},
    diag_kws={'alpha': 0.7, 'linewidth': 2}
)
plt.tight_layout()
plt.show()

## Stylistic differences statistics

In [ ]:
total_tokens = df['tokens_count_clean'].sum()
male_tokens = male_df['tokens_count_clean'].sum()
female_tokens = female_df['tokens_count_clean'].sum()

print(f"Total tokens in dataset: {total_tokens:,}")
print(f"Male tokens: {male_tokens:,}")
print(f"Female tokens: {female_tokens:,}")

### char n-grams

In [ ]:
vect_test = CountVectorizer(
    analyzer='char',
    ngram_range=(3,3),
    lowercase=True,
    min_df=10
)
x_test = vect_test.fit_transform(df['clean_tokens'].apply(lambda tokens: ' '.join(tokens)))
total_ngrams = len(vect_test.get_feature_names_out())
print(f"Total n-grams with min_df=10: {total_ngrams:,}")

we will use max features to limit number of n-grams

In [ ]:
vect = CountVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    lowercase=True,
    min_df=10,
    max_features=int(total_ngrams*0.7)
)

x_char: np.ndarray = vect.fit_transform(df['clean_tokens'].apply(lambda tokens: '_'.join(tokens)))  # type: ignore
feats = np.array(vect.get_feature_names_out())
freq = {}
for g in df['gender_label'].unique():
    subX = x_char[df['gender_label'].values == g]
    counts = np.asarray(subX.sum(axis=0)).ravel()
    total = counts.sum()
    freq[g] = (counts / max(total, 1)) * 10000

out = pd.DataFrame(freq, index=feats)
out['total_count'] = np.asarray(x_char.sum(axis=0)).ravel()
out.index.name = f'char_3_5_ngram'
out.head()

In [ ]:
top_n = 20

male_top = out.nlargest(top_n, 'M')[['M', 'total_count']]
female_top = out.nlargest(top_n, 'F')[['F', 'total_count']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(range(len(male_top)), male_top['M'], color='steelblue')
axes[0].set_yticks(range(len(male_top)))
axes[0].set_yticklabels(male_top.index)
axes[0].set_xlabel('Frequency per 10k tokens')
axes[0].set_title(f'Top {top_n} char n-grams - Male')
axes[0].invert_yaxis()

axes[1].barh(range(len(female_top)), female_top['F'], color='red')
axes[1].set_yticks(range(len(female_top)))
axes[1].set_yticklabels(female_top.index)
axes[1].set_xlabel('Frequency per 10k tokens')
axes[1].set_title(f'Top {top_n} char n-grams - Female')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
d = out[out['total_count'] > 150].copy()
A = d.get('M', 0.0).astype(float)  # type: ignore
B = d.get('F', 0.0).astype(float)  # type: ignore
d['log_ratio'] = np.log((A + 0.5) / (B + 0.5))
d['log_ratio_abs'] = d['log_ratio'].abs()
d.head()

In [ ]:
sel = d.sort_values('log_ratio_abs', ascending=False).head(30)
index = sel.index.str[:20] + '...'
index = index.tolist()
sel['color'] = ['blue' if val > 0 else 'red' for val in sel['log_ratio']]
plt.barh(range(len(sel)), sel['log_ratio'], color=sel['color'].tolist())
plt.yticks(range(len(sel)), index)

plt.xlabel(f'log((ppm M+0.5)/(ppm F+0.5))', fontsize=12)
plt.title(f'Word n-grams: F (red) vs M (blue)', fontsize=14, fontweight='bold')

### word n-grams

check how much ngrams for 3 grams we can make with min_df=10

In [ ]:
vect_test = CountVectorizer(
    analyzer='word',
    ngram_range=(3,3),
    min_df=10,
)
x_word = vect_test.fit_transform(df['dirty_tokens'].apply(lambda tokens: ' '.join(tokens)))
total_ngrams = len(vect_test.get_feature_names_out())
print(f"Total n-grams with min_df=10: {total_ngrams:,}")

In [ ]:
vect = CountVectorizer(
    analyzer='word',
    ngram_range=(3,5),
    min_df=10,
    max_features=int(total_ngrams*0.7)
)

x_word: np.ndarray = vect.fit_transform(df['dirty_tokens'].apply(lambda tokens: ' '.join(tokens)))  # type: ignore
feats = np.array(vect.get_feature_names_out())

freq_M = []
freq_F = []
count_M = []
count_F = []

for g in df['gender_label'].unique():
    subX = x_word[df['gender_label'].values == g]
    counts = np.asarray(subX.sum(axis=0)).ravel()
    total = counts.sum()
    freq_per_10k = (counts / max(total, 1)) * 10000
    
    if g == 'M':
        freq_M = freq_per_10k
        count_M = counts
    else:
        freq_F = freq_per_10k
        count_F = counts

word_gram_freq = pd.DataFrame({
    'M': freq_M,
    'F': freq_F,
    'M_count': count_M,
    'F_count': count_F
}, index=feats)

word_gram_freq['total_count'] = np.asarray(x_word.sum(axis=0)).ravel()
word_gram_freq.index.name = f'word_3_5_ngram'
word_gram_freq.head()

top char ngrams for each gender 

In [ ]:
top_n = 20

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

male_top = word_gram_freq[word_gram_freq['M'] > 0].nlargest(top_n, 'M')[['M', 'total_count']]
axes[0].barh(range(len(male_top)), male_top['M'], color='steelblue')
axes[0].set_yticks(range(len(male_top)))
axes[0].set_yticklabels(male_top.index)
axes[0].set_xlabel('Frequency per 10k tokens')
axes[0].set_title(f'Top {top_n} word n-grams - Male')
axes[0].invert_yaxis()

female_top = word_gram_freq[word_gram_freq['F'] > 0].nlargest(top_n, 'F')[['F', 'total_count']]
axes[1].barh(range(len(female_top)), female_top['F'], color='red')
axes[1].set_yticks(range(len(female_top)))
axes[1].set_yticklabels(female_top.index)
axes[1].set_xlabel('Frequency per 10k tokens')
axes[1].set_title(f'Top {top_n} word n-grams - Female')
axes[1].invert_yaxis()

Analyzing ngrams Which are more characteristic of each gender using log ratio

In [ ]:
d = out.copy()
A = d.get('M', 0.0).astype(float)  # type: ignore
B = d.get('F', 0.0).astype(float)  # type: ignore
d['log_ratio'] = np.log((A + 0.5) / (B + 0.5))
d['log_ratio_abs'] = d['log_ratio'].abs()

In [ ]:
sel = d.sort_values('log_ratio_abs', ascending=False).head(30)
index = sel.index.str[:20] + '...'
index = index.tolist()
sel['color'] = ['blue' if val > 0 else 'red' for val in sel['log_ratio']]
plt.barh(range(len(sel)), sel['log_ratio'], color=sel['color'].tolist())
plt.yticks(range(len(sel)), index)

plt.xlabel(f'log((ppm M+0.5)/(ppm F+0.5))', fontsize=12)
plt.title(f'Word n-grams: F (red) vs M (blue)', fontsize=14, fontweight='bold')

### word skip ngrams

In [ ]:
def make_skip_analyzer(n: tuple[int,int], k: int):
    def analyzer(text: str| list[str]):
        if isinstance(text, list):
            toks = text
        elif isinstance(text, str):
            toks = text.split()
        else:
            raise ValueError("Input should be a string or a list of strings.")
        for i in range(n[0], n[1] + 1):
            for g in nltk.skipgrams(toks, n=i, k=k):
                yield " ".join(g)
    return analyzer


In [ ]:
vect_test = CountVectorizer(
    analyzer=make_skip_analyzer(n=(2,2), k=1),
    min_df=10,
)
x_word = vect_test.fit_transform(df['dirty_tokens'].apply(lambda tokens: ' '.join(tokens)))
total_skip_ngrams = len(vect_test.get_feature_names_out())
print(f"Total skip n-grams with min_df=10: {total_skip_ngrams:,}")

In [ ]:
vect_skip = CountVectorizer(
    analyzer=make_skip_analyzer(n=(3,4), k=2),
    min_df=10,
    max_features=int(total_skip_ngrams*0.7)
)
x_skip: np.ndarray = vect_skip.fit_transform(df['dirty_tokens']) # type: ignore
feats = np.array(vect_skip.get_feature_names_out())
freq_M = []
freq_F = []
count_M = []
count_F = []
for g in df['gender_label'].unique():
    subX = x_skip[df['gender_label'].values == g]
    counts = np.asarray(subX.sum(axis=0)).ravel()
    total = counts.sum()
    freq_per_10k = (counts / max(total, 1)) * 10000
    if g == 'M':
        freq_M = freq_per_10k
        count_M = counts
    else:
        freq_F = freq_per_10k
        count_F = counts
word_skip_gram_freq = pd.DataFrame({
    'M': freq_M,
    'F': freq_F,
    'M_count': count_M,
    'F_count': count_F
}, index=feats)
word_skip_gram_freq['total_count'] = np.asarray(x_skip.sum(axis=0)).ravel()
word_skip_gram_freq.index.name = f'skip_word_3_5_ngrams_skip_1'
word_skip_gram_freq.head()

In [ ]:
top_n = 20

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

male_top = word_skip_gram_freq[word_skip_gram_freq['M'] > 0].nlargest(top_n, 'M')[['M', 'total_count']]
axes[0].barh(range(len(male_top)), male_top['M'], color='steelblue')
axes[0].set_yticks(range(len(male_top)))
axes[0].set_yticklabels(male_top.index)
axes[0].set_xlabel('Frequency per 10k tokens')
axes[0].set_title(f'Top {top_n} word n-grams - Male')
axes[0].invert_yaxis()

female_top = word_gram_freq[word_gram_freq['F'] > 0].nlargest(top_n, 'F')[['F', 'total_count']]
axes[1].barh(range(len(female_top)), female_top['F'], color='red')
axes[1].set_yticks(range(len(female_top)))
axes[1].set_yticklabels(female_top.index)
axes[1].set_xlabel('Frequency per 10k tokens')
axes[1].set_title(f'Top {top_n} word n-grams - Female')
axes[1].invert_yaxis()

# Statistics tests by groups

#### mann whitney u test for numerical features

Nonparametric tests for numerical characteristics Mann–Whitney U

Null hypothesis H0: the distributions of the characteristic in groups M and F are identical.

For each numerical column, calculates the U statistic and p-value, and marks significant characteristics at p < 0.05.

In [ ]:
numerical_features = df.select_dtypes(include=['number']).columns.tolist()

mann_whitney_results = []
for feature in numerical_features:
    male_values = male_df[feature].dropna()
    female_values = female_df[feature].dropna()
    
    statistic, p_value = stats.mannwhitneyu(male_values, female_values, alternative='two-sided')
    
    mann_whitney_results.append({
        'feature': feature,
        'statistic': statistic,
        'p_value': round(p_value, 4),
        'significant': p_value < 0.05
    })
    
mann_whitney_df = pd.DataFrame(mann_whitney_results)
mann_whitney_df = mann_whitney_df.sort_values(['p_value', 'statistic'], ascending=[True, False])
print(mann_whitney_df.to_string(index=False))

#### chi square test for categorical features

Categorical comparisons of words using the Chi-square test whether the use of a particular word depends 
on gender

Null hypothesis H0 the use of w does not depend on gender

p < 0.05 differences in usage are statistically significant

Cramer V is used to assess the strength of the relationship between word usage and user gender.

In [ ]:
top_n_tokens = 100
male_top_tokens = male_word_freq.most_common(top_n_tokens)
female_top_tokens = female_word_freq.most_common(top_n_tokens)

all_top_words = set([word for word, _ in male_top_tokens] + [word for word, _ in female_top_tokens])

chi2_results = []
for word in all_top_words:
    male_count = male_word_freq[word]
    female_count = female_word_freq[word]
    
    contingency_table = np.array([
        [male_count, male_tokens - male_count],
        [female_count, female_tokens - female_count]
    ])
    
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
    
    cramers_v = stats.contingency.association(contingency_table, method='cramer')
    
    chi2_results.append({
        'word': word,
        'male_count': male_count,
        'female_count': female_count,
        'chi2': chi2,
        'p_value': p_value, # type: ignore
        'cramers_v': cramers_v,
        'significant': p_value <= 0.1 # type: ignore
    })
chi2_df = pd.DataFrame(chi2_results)
chi2_df = chi2_df.sort_values(['p_value', 'chi2'], ascending=[True, False])
chi2_df['p_value'] = chi2_df['p_value'].round(4)
print(chi2_df[ (chi2_df['significant'] == True) & (chi2_df['cramers_v'] > 0.05) ].to_string(index=False))

## statistic for word n grams

Categorical comparisons of words using the Chi-square test whether the use of a particular ngram depends 
on gender

Null hypothesis H0 the use of w does not depend on gender

p < 0.05 differences in usage are statistically significant

Cramer V is used to assess the strength of the relationship between word usage and user gender.

In [ ]:
top_n_word_grams = 100
base_word_ngram = word_gram_freq.query('total_count > 0')

male_top_word_grams = (base_word_ngram.sort_values('M_count', ascending=False)
                 .head(top_n_word_grams)['M_count']
                 .astype(int)
                 .to_dict())

female_top_word_grams = (base_word_ngram.sort_values('F_count', ascending=False)
                   .head(top_n_word_grams)['F_count']
                   .astype(int)
                   .to_dict())

all_top_word_grams = set(male_top_word_grams) | set(female_top_word_grams)

chi2_results = []
for ngram in all_top_word_grams:
    male_count = 0
    female_count = 0
    if ngram in male_top_word_grams:
        male_count = male_top_word_grams[ngram]
    if ngram in female_top_word_grams:
        female_count = female_top_word_grams[ngram]
    contingency_table = np.array([
        [male_count, male_tokens - male_count],
        [female_count, female_tokens - female_count]
    ])

    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table, correction=False)

    cramers_v = stats.contingency.association(contingency_table, method='cramer')

    chi2_results.append({
        'ngram': ngram,
        'male_count': male_count,
        'female_count': female_count,
        'chi2': chi2,
        'p_value': p_value,
        'cramers_v': cramers_v,
        'significant': p_value < 0.05 # type: ignore
    })

chi2_df = pd.DataFrame(chi2_results)

mask_ok = (chi2_df['ngram'].map(lambda w: True))

chi2_df = chi2_df[mask_ok]

chi2_df = chi2_df.sort_values(['p_value', 'chi2'], ascending=[True, False])

chi2_df['p_value'] = chi2_df['p_value'].round(4)
print(chi2_df.loc[ (chi2_df['significant'] == True) & (chi2_df['cramers_v'] > 0.05) ].to_string(index=False))